# Exploratory Data Analysis — EUKLEMS Growth Accounts

This notebook constitutes the first analytical step of the project and is diagnostic in character: its purpose is to establish the structure and coverage of the EUKLEMS Growth Accounts dataset and to prepare it for the modelling steps in the downstream notebooks. The analysis proceeds in five stages.
1) The required libraries are imported, path constants are configured, and the dataset is loaded from disk; a first preview confirms that the file has been read correctly and reports the overall dimensions of the panel.
2) The four basic dimensions of the panel — countries, industries, years, and variables — are enumerated, providing an upper bound on the size of a fully balanced panel and confirming that the NACE Rev. 2 industry classification is consistent with the ICIO table used in Notebook 4.
3) The raw long-format data are inspected for missing values and data types without modifying any observations; only the `value` column carries missing entries (1,001,825 out of 2,467,044 rows, i.e. 40.61%), while all identifier columns are fully populated.
4) The long-format panel is pivoted into wide format so that each of the 68 variables occupies a dedicated column; the resulting `df_euklem_adjusted` dataframe uses five index columns (`geo_code`, `geo_name`, `nace_r2_code`, `nace_r2_name`, `year`) and contains 45,267 rows × 73 columns, and is the canonical representation used in all subsequent notebooks.
5) A country-specific subset is extracted by filtering `df_euklem_adjusted` to rows where `geo_code` equals `'IT'`; the shape of `df_italy` and the number of available variable columns directly determine the sample available for the Italian productivity analysis in Notebooks 3 and 4.


## 1. Data and Libraries Loading

The analysis starts by importing the necessary libraries and loading the EUKLEMS Growth Accounts dataset. 

In [31]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure display options so wide DataFrames are not silently truncated
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Set plotting style and defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Set path constants and ensure output directories exist
DATA_RAW  = Path('../data/raw')
FIG_DIR   = Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load the raw panel dataset
df_euklem = pd.read_csv(DATA_RAW / 'growth accounts.csv')
print("EUKLEMS dataset loaded successfully")
print(f"The dataset has {df_euklem.shape[0]} rows and {df_euklem.shape[1]} columns.")
print("Displaying the first 5 rows of the dataset:")
print(df_euklem.head(5))


EUKLEMS dataset loaded successfully
The dataset has 2467044 rows and 8 columns.
Displaying the first 5 rows of the dataset:
   Unnamed: 0 nace_r2_code geo_code  year                       nace_r2_name  \
0           4            A       AT  1995  Agriculture, forestry and fishing   
1           5            A       AT  1995  Agriculture, forestry and fishing   
2          37            A       AT  1995  Agriculture, forestry and fishing   
3          38            A       AT  1995  Agriculture, forestry and fishing   
4          39            A       AT  1995  Agriculture, forestry and fishing   

  geo_name         var     value  
0  Austria         LAB 3669.5984  
1  Austria         CAP  193.1368  
2  Austria      CAP_QI   97.5399  
3  Austria   CAPICT_QI   29.3106  
4  Austria  CAPNICT_QI   99.2779  


## 2. Dimensional Analysis

The EUKLEMS dataset is structured as a long panel in which each row corresponds to a single value of a single variable for a single country, industry, and year. Before any substantive analysis it is necessary to establish the four basic dimensions of the dataset — the number of unique countries, industries, years, and variables — since these determine the theoretical maximum size of a balanced panel and therefore set an upper bound on the feasibility of the modelling step. The cell below also prints the full lists of NACE codes and variable names, which are required to cross-check coverage against the OECD taxonomy used in the ICIO table.


In [ ]:
# Count unique entities along each dimension to establish the panel scope
n_countries  = df_euklem['geo_code'].nunique()
n_sectors = df_euklem['nace_r2_code'].nunique()
n_vars = df_euklem['var'].nunique()

# Report basic dimensions, entity lists, and full variable catalogue
print(f"Total observations: {len(df_euklem):,}")
print(f"Countries: {n_countries}")
print(f"Sectors: {n_sectors}")
print(f"Variables: {n_vars}")


Total observations : 2,467,044
Countries: 34
Sectors: 58
Variables: 68


The output above establishes the scope of the raw data. Note the year range and variable count carefully — both will constrain what is feasible in the modelling step. The industry codes follow NACE Rev. 2, the same classification used in the ICIO table, which is a prerequisite for the cross-dataset merge in notebook 4. Any discrepancies in code granularity between the two datasets will need to be resolved there.

## 3. Missing Values and Data Types

Before any transformation the raw long-format data are inspected for missing values. 


In [26]:
# Count missing values per column in the raw dataset
for col in df_euklem.columns:
    print(f"The '{col}' column has {int(df_euklem[col].isna().sum())} missing values and has data type {df_euklem[col].dtype}.")


The 'Unnamed: 0' column has 0 missing values and has data type int64.
The 'nace_r2_code' column has 0 missing values and has data type object.
The 'geo_code' column has 0 missing values and has data type object.
The 'year' column has 0 missing values and has data type int64.
The 'nace_r2_name' column has 0 missing values and has data type object.
The 'geo_name' column has 0 missing values and has data type object.
The 'var' column has 0 missing values and has data type object.
The 'value' column has 1001825 missing values and has data type float64.


The dataset does not present missing values, hence the pivoting step can proceed without any imputation or row-dropping. The data types of the columns are also reported.

## 4. Data Reconstruction

The EUKLEMS dataset is delivered in long format, with one observation per row identified by `geo_code`, `geo_name`, `nace_r2_code`, `nace_r2_name`, `year`, and `var`. For all downstream modelling steps a wide-format panel is required, in which each variable occupies a dedicated column and each row corresponds to a unique country-industry-year combination. The pivot below performs this transformation using five index columns (`geo_code`, `geo_name`, `nace_r2_code`, `nace_r2_name`, `year`) to preserve both codes and human-readable names. The result is stored in `df_euklem_adjusted` (45,267 rows × 73 columns: 5 index + 68 variable columns) and is the canonical representation used in all subsequent notebooks.


In [29]:
# Pivot from long to wide: each variable becomes a column indexed by country, industry, and year
df_euklem_adjusted = (
    df_euklem
    .pivot_table(index=['geo_code', 'geo_name', 'nace_r2_code', 'nace_r2_name', 'year'],
                 columns='var',
                 values='value')
    .reset_index()
)
df_euklem_adjusted.columns.name = None

print(f"Wide-format shape: {df_euklem_adjusted.shape[0]:,} rows × {df_euklem_adjusted.shape[1]} columns")
print(f"Index columns: geo_code, geo_name, nace_r2_code, nace_r2_name, year")
print(f"Variable columns: {df_euklem_adjusted.shape[1] - 5}  (one per unique 'var' value)")
print("\nFirst 5 rows (index columns + first 3 variable columns):")
print(df_euklem_adjusted.iloc[:5, :8])


Wide-format shape: 45,267 rows × 73 columns
Index columns: geo_code, geo_name, nace_r2_code, nace_r2_name, year
Variable columns: 68  (one per unique 'var' value)

First 5 rows (index columns + first 3 variable columns):
  geo_code geo_name nace_r2_code                       nace_r2_name  year  \
0       AT  Austria            A  Agriculture, forestry and fishing  1995   
1       AT  Austria            A  Agriculture, forestry and fishing  1996   
2       AT  Austria            A  Agriculture, forestry and fishing  1997   
3       AT  Austria            A  Agriculture, forestry and fishing  1998   
4       AT  Austria            A  Agriculture, forestry and fishing  1999   

       CAP  CAPEconComp_QI  CAPICT_QI  
0 193.1368        246.7640    29.3106  
1 180.3851        238.2784    32.0632  
2 185.7125        236.5965    34.8025  
3 180.5240        233.0708    38.1270  
4 181.8660        231.6847    42.1616  


## 5. Italian Subset

Italy is the focal country of the project. The cell below filters `df_euklem_adjusted` to rows where `geo_code` equals `'IT'` and stores the result in `df_italy`. The shape of this subset — number of sector-year observations and the 68 available variable columns — directly determines the maximum sample available for the modelling steps in Notebooks 3 and 4.


In [ ]:
# Extract Italy-only rows from the wide-format panel
df_italy = df_euklem_adjusted[df_euklem_adjusted['geo_code'] == 'IT'].reset_index(drop=True)

print(f"Italian subset shape: {df_italy.shape[0]} rows × {df_italy.shape[1]} columns")
print(f"Sector-year observations: {df_italy.shape[0]}")
print(f"Unique sectors: {df_italy['nace_r2_code'].nunique()}")
print(f"Unique years: {df_italy['year'].nunique()} ({int(df_italy['year'].min())}–{int(df_italy['year'].max())})")
print(f"Variable columns: {df_italy.shape[1] - 5}")
print("\nFirst 5 rows (index columns + first 3 variable columns):")
print(df_italy.iloc[:5, :8].to_string())


Italian subset shape    : 1514 rows × 73 columns
Sector-year observations: 1514
Unique sectors          : 58
Unique years            : 27  (1995–2021)
Variable columns        : 68

First 5 rows (index columns + first 3 variable columns):
  geo_code geo_name nace_r2_code                       nace_r2_name  year        CAP  CAPEconComp_QI  CAPICT_QI
0       IT    Italy            A  Agriculture, forestry and fishing  1995  8190.4966         95.5189    55.0545
1       IT    Italy            A  Agriculture, forestry and fishing  1996  9611.9316         96.6618    57.1952
2       IT    Italy            A  Agriculture, forestry and fishing  1997  9807.8379         96.5492    55.5741
3       IT    Italy            A  Agriculture, forestry and fishing  1998 10963.9258         95.3859    62.3029
4       IT    Italy            A  Agriculture, forestry and fishing  1999 12121.7764         94.7983    66.5686
